In [3]:
import pandas as pd 
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os
import sys
import argparse

In [4]:
irrigation_data = pd.read_csv("playground-series-s6e4/train.csv")
irrigation_data.head()

,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,...,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,...,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,...,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,...,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low
3,3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,...,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium
4,4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,...,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low


In [5]:
irrigation_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 630000 entries, 0 to 629999
Data columns (total 21 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   id                       630000 non-null  int64  
 1   Soil_Type                630000 non-null  object 
 2   Soil_pH                  630000 non-null  float64
 3   Soil_Moisture            630000 non-null  float64
 4   Organic_Carbon           630000 non-null  float64
 5   Electrical_Conductivity  630000 non-null  float64
 6   Temperature_C            630000 non-null  float64
 7   Humidity                 630000 non-null  float64
 8   Rainfall_mm              630000 non-null  float64
 9   Sunlight_Hours           630000 non-null  float64
 10  Wind_Speed_kmh           630000 non-null  float64
 11  Crop_Type                630000 non-null  object 
 12  Crop_Growth_Stage        630000 non-null  object 
 13  Season                   630000 non-null  object 
 14  Irri

In [6]:
irrigation_data.nunique()

id                         630000
Soil_Type                       4
Soil_pH                       341
Soil_Moisture                5223
Organic_Carbon                131
Electrical_Conductivity       341
Temperature_C                2934
Humidity                     6475
Rainfall_mm                 19308
Sunlight_Hours                701
Wind_Speed_kmh               1935
Crop_Type                       6
Crop_Growth_Stage               4
Season                          3
Irrigation_Type                 4
Water_Source                    4
Field_Area_hectare           1466
Mulching_Used                   2
Previous_Irrigation_mm      10110
Region                          5
Irrigation_Need                 3
dtype: int64

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score
from catboost import CatBoostClassifier
from sklearn.preprocessing import LabelEncoder

# Encode target
le = LabelEncoder()
irrigation_data['Irrigation_Need_encoded'] = le.fit_transform(irrigation_data['Irrigation_Need'])

# Identify categorical features
cat_features = ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']

# Features and target
X = irrigation_data.drop(['id', 'Irrigation_Need', 'Irrigation_Need_encoded'], axis=1)
y = irrigation_data['Irrigation_Need_encoded']

# Split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Train CatBoost
model = CatBoostClassifier(iterations=1000, learning_rate=0.1, depth=6, cat_features=cat_features, verbose=100)
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_val)

# Balanced accuracy
bal_acc = balanced_accuracy_score(y_val, y_pred)
print(f'Balanced Accuracy: {bal_acc:.4f}')

0:	learn: 0.9191729	total: 483ms	remaining: 8m 2s
100:	learn: 0.0634493	total: 39.3s	remaining: 5m 49s
200:	learn: 0.0610375	total: 1m 20s	remaining: 5m 19s
300:	learn: 0.0596901	total: 2m	remaining: 4m 40s
400:	learn: 0.0587047	total: 2m 40s	remaining: 3m 59s
500:	learn: 0.0578408	total: 3m 22s	remaining: 3m 21s
600:	learn: 0.0569745	total: 4m 3s	remaining: 2m 41s
700:	learn: 0.0561477	total: 4m 44s	remaining: 2m 1s
800:	learn: 0.0554240	total: 5m 26s	remaining: 1m 21s
900:	learn: 0.0548374	total: 6m 6s	remaining: 40.3s
999:	learn: 0.0541817	total: 6m 46s	remaining: 0us
Balanced Accuracy: 0.9642


In [9]:
# Load test data
test_data = pd.read_csv("playground-series-s6e4/test.csv")

# Predict on test
test_pred_encoded = model.predict(test_data.drop('id', axis=1))

# Decode predictions
test_pred = le.inverse_transform(test_pred_encoded)

# Create submission
submission = pd.DataFrame({'id': test_data['id'], 'Irrigation_Need': test_pred})
submission.to_csv('submission_new.csv', index=False)
print('Submission saved as submission.csv')

Submission saved as submission.csv


/home/prinshu/Desktop/Machine_learning/.venv/lib/python3.10/site-packages/sklearn/preprocessing/_label.py:151: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [10]:
test_data = pd.read_csv("playground-series-s6e4/test.csv")
submission_new = pd.read_csv('playground-series-s6e4/submission_new.csv')

test_data.head()

,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region
0,630000,Silt,6.36,26.19,0.59,2.81,17.83,30.24,1533.38,5.40,3.00,Maize,Sowing,Rabi,Canal,River,13.59,Yes,47.48,West
1,630001,Clay,5.87,9.88,1.18,3.26,21.18,78.07,576.05,7.22,15.88,Cotton,Sowing,Rabi,Drip,Reservoir,6.12,Yes,56.43,South
2,630002,Sandy,6.22,26.55,0.96,0.85,26.87,60.35,545.30,9.43,2.63,Wheat,Sowing,Kharif,Sprinkler,Reservoir,3.11,Yes,20.00,East
3,630003,Clay,7.68,53.58,0.83,0.55,41.74,36.05,1211.03,6.69,1.86,Maize,Harvest,Rabi,Canal,Groundwater,2.27,No,102.99,North
4,630004,Loamy,5.23,59.02,0.54,2.11,41.08,52.47,1321.91,4.11,5.71,Cotton,Sowing,Kharif,Canal,Groundwater,12.39,Yes,13.33,Central


In [11]:
submission_new.head()

,id,Irrigation_Need
0,630000,Low
1,630001,Low
2,630002,Low
3,630003,Low
4,630004,Low


In [8]:
# Load additional training data
additional_data = pd.read_csv("playground-series-s6e4/irrigation_prediction.csv")

# Combine with original data
combined_data = pd.concat([irrigation_data, additional_data], ignore_index=True)

# Encode target again
combined_data['Irrigation_Need_encoded'] = le.fit_transform(combined_data['Irrigation_Need'])

# Features and target
X_combined = combined_data.drop(['id', 'Irrigation_Need', 'Irrigation_Need_encoded'], axis=1)
y_combined = combined_data['Irrigation_Need_encoded']

# Split
X_train, X_val, y_train, y_val = train_test_split(X_combined, y_combined, test_size=0.2, random_state=42, stratify=y_combined)

# Train improved model with more iterations and better params
model_improved = CatBoostClassifier(iterations=2000, learning_rate=0.05, depth=8, cat_features=cat_features, verbose=200)
model_improved.fit(X_train, y_train)

# Predict
y_pred = model_improved.predict(X_val)

# Balanced accuracy and precision
from sklearn.metrics import precision_score
bal_acc = balanced_accuracy_score(y_val, y_pred)
prec = precision_score(y_val, y_pred, average='macro')
print(f'Balanced Accuracy: {bal_acc:.4f}')
print(f'Precision (macro): {prec:.4f}')

0:	learn: 1.0061490	total: 652ms	remaining: 21m 42s
200:	learn: 0.0608167	total: 2m 2s	remaining: 18m 13s
400:	learn: 0.0589202	total: 3m 59s	remaining: 15m 56s
600:	learn: 0.0572016	total: 6m 16s	remaining: 14m 36s
800:	learn: 0.0555884	total: 8m 32s	remaining: 12m 47s
1000:	learn: 0.0541268	total: 10m 48s	remaining: 10m 47s
1200:	learn: 0.0528035	total: 13m 1s	remaining: 8m 39s
1400:	learn: 0.0515455	total: 15m 8s	remaining: 6m 28s
1600:	learn: 0.0503076	total: 17m 1s	remaining: 4m 14s
1800:	learn: 0.0490728	total: 18m 53s	remaining: 2m 5s
1999:	learn: 0.0478968	total: 20m 40s	remaining: 0us
Balanced Accuracy: 0.9628
Precision (macro): 0.9803


In [9]:
# Load test data
test_data = pd.read_csv("playground-series-s6e4/test.csv")

# Predict on test with improved model
test_pred_encoded = model_improved.predict(test_data.drop('id', axis=1))

# Decode predictions
test_pred = le.inverse_transform(test_pred_encoded)

# Create submission
submission_improved = pd.DataFrame({'id': test_data['id'], 'Irrigation_Need': test_pred})
submission_improved.to_csv('submission_improved.csv', index=False)
print('Improved submission saved as submission_improved.csv')

Improved submission saved as submission_improved.csv


/home/prinshu/Desktop/Machine_learning/.venv/lib/python3.10/site-packages/sklearn/preprocessing/_label.py:151: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [10]:
# Additional metrics
from sklearn.metrics import recall_score, f1_score, classification_report

recall = recall_score(y_val, y_pred, average='macro')
f1 = f1_score(y_val, y_pred, average='macro')

print(f'Recall (macro): {recall:.4f}')
print(f'F1 Score (macro): {f1:.4f}')
print('\nClassification Report:')
print(classification_report(y_val, y_pred, target_names=le.classes_))

Recall (macro): 0.9628
F1 Score (macro): 0.9713

Classification Report:
              precision    recall  f1-score   support

        High       0.97      0.92      0.94      4269
         Low       0.99      1.00      0.99     75156
      Medium       0.99      0.98      0.98     48575

    accuracy                           0.99    128000
   macro avg       0.98      0.96      0.97    128000
weighted avg       0.99      0.99      0.99    128000



In [11]:
# Predict on additional data to check generalization
additional_pred_encoded = model_improved.predict(additional_data.drop(['Irrigation_Need'], axis=1))
additional_pred = le.inverse_transform(additional_pred_encoded)

# Compare with actual
actual = additional_data['Irrigation_Need']
accuracy_additional = (additional_pred == actual).mean()
print(f'Accuracy on additional data: {accuracy_additional:.4f}')

# Balanced accuracy on additional
bal_acc_additional = balanced_accuracy_score(le.transform(actual), additional_pred_encoded)
print(f'Balanced Accuracy on additional data: {bal_acc_additional:.4f}')

Accuracy on additional data: 0.9948
Balanced Accuracy on additional data: 0.9798


/home/prinshu/Desktop/Machine_learning/.venv/lib/python3.10/site-packages/sklearn/preprocessing/_label.py:151: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [12]:
# Try LightGBM for comparison
from lightgbm import LGBMClassifier

# Encode categorical for LightGBM
X_combined_lgb = X_combined.copy()
for col in cat_features:
    X_combined_lgb[col] = X_combined_lgb[col].astype('category')

# Split again
X_train_lgb, X_val_lgb, y_train_lgb, y_val_lgb = train_test_split(X_combined_lgb, y_combined, test_size=0.2, random_state=42, stratify=y_combined)

# Train LightGBM
lgb_model = LGBMClassifier(n_estimators=1000, learning_rate=0.05, max_depth=8, verbose=-1)
lgb_model.fit(X_train_lgb, y_train_lgb, categorical_feature=cat_features)

# Predict
y_pred_lgb = lgb_model.predict(X_val_lgb)

# Metrics
bal_acc_lgb = balanced_accuracy_score(y_val_lgb, y_pred_lgb)
prec_lgb = precision_score(y_val_lgb, y_pred_lgb, average='macro')
print(f'LightGBM - Balanced Accuracy: {bal_acc_lgb:.4f}, Precision: {prec_lgb:.4f}')

# If better, use for prediction
if prec_lgb > prec:
    print('LightGBM has better precision, using it for submission.')
    test_pred_encoded_lgb = lgb_model.predict(test_data.drop('id', axis=1).astype({col: 'category' for col in cat_features}))
    test_pred_lgb = le.inverse_transform(test_pred_encoded_lgb)
    submission_lgb = pd.DataFrame({'id': test_data['id'], 'Irrigation_Need': test_pred_lgb})
    submission_lgb.to_csv('submission_lgb.csv', index=False)
else:
    print('CatBoost is better.')

LightGBM - Balanced Accuracy: 0.9627, Precision: 0.9788
CatBoost is better.


In [14]:
# Ensemble CatBoost and LightGBM
from scipy.stats import mode
import numpy as np

# Get predictions from both models
cat_pred = model_improved.predict(X_val)
lgb_pred = lgb_model.predict(X_val_lgb)

# Stack predictions
pred_stack = np.column_stack((cat_pred, lgb_pred))

# Ensemble by majority vote
ensemble_pred = mode(pred_stack, axis=1)[0].flatten()

# Metrics for ensemble
bal_acc_ens = balanced_accuracy_score(y_val, ensemble_pred)
prec_ens = precision_score(y_val, ensemble_pred, average='macro')
print(f'Ensemble - Balanced Accuracy: {bal_acc_ens:.4f}, Precision: {prec_ens:.4f}')

# If better, use for submission
if prec_ens > prec:
    print('Ensemble has better precision, using it for submission.')
    cat_test_pred = model_improved.predict(test_data.drop('id', axis=1))
    lgb_test_pred = lgb_model.predict(test_data.drop('id', axis=1).astype({col: 'category' for col in cat_features}))
    test_pred_stack = np.column_stack((cat_test_pred, lgb_test_pred))
    ensemble_test_pred = mode(test_pred_stack, axis=1)[0].flatten()
    test_pred_ens = le.inverse_transform(ensemble_test_pred)
    submission_ens = pd.DataFrame({'id': test_data['id'], 'Irrigation_Need': test_pred_ens})
    submission_ens.to_csv('submission_ensemble.csv', index=False)
else:
    print('CatBoost is still better.')

Ensemble - Balanced Accuracy: 0.9644, Precision: 0.9787
CatBoost is still better.
